In [ ]:
dbutils.widgets.text(
    "catalog",
    "predictivemaintenance_dev",
    "Target Catalog"
)

CATALOG = dbutils.widgets.get("catalog")

print(f"Target catalog: {CATALOG}")
#------------------------------------------------#
# Optional starting version for Delta streaming
dbutils.widgets.text(
    "starting_version",
    "",
    "Starting Delta Version"
)

STARTING_VERSION = dbutils.widgets.get("starting_version").strip()

In [ ]:
from pyspark.sql import functions as F
# Identify the deployment environment
dbutils.widgets.text("deployment_target", "dev")
DEPLOYMENT_TARGET = dbutils.widgets.get("deployment_target")

# Expected catalog for each environment
EXPECTED_CATALOGS = {
    "dev": "predictivemaintenance_dev",
    "prod": "predictivemaintenancecatalog"
}

# Prevent accidental writes to the wrong catalog
if DEPLOYMENT_TARGET not in EXPECTED_CATALOGS:
    raise ValueError(f"Unknown deployment target: {DEPLOYMENT_TARGET}")

if CATALOG != EXPECTED_CATALOGS[DEPLOYMENT_TARGET]:
    raise ValueError(
        f"Catalog {CATALOG} does not match "
        f"deployment target {DEPLOYMENT_TARGET}"
    )

print(f"Deployment target: {DEPLOYMENT_TARGET}")
print(f"Catalog: {CATALOG}")

In [0]:
from pyspark.sql import functions as F

# Environment-specific tables
landing_table = f"{CATALOG}.raw_data.landing_testdata"

bronze_table = (
    f"{CATALOG}.bronze_data_not_quality_checked.streaming"
)

# Environment-specific checkpoint
landing_path = f"/Volumes/{CATALOG}/raw_data/landing"

data_checkpoint_location = (
    f"{landing_path}/_checkpoints/streaming_data_uc_landing_v1"
)


# Read commits from the registered Delta table, NOT the volume path.
# Version 0 contains the initial 2010 rows already handled separately.
# IMPORTANT: the first run must happen AFTER the table has a version 1 commit.
# 'startingVersion' is used only when this new checkpoint initializes.
stream_reader = spark.readStream.format("delta")

if STARTING_VERSION:
    if not STARTING_VERSION.isdigit():
        raise ValueError("starting_version must be a nonnegative integer")

    stream_reader = stream_reader.option(
        "startingVersion", int(STARTING_VERSION)
    )

raw_data_streaming = stream_reader.table(landing_table)


# Add ingestion metadata
data_bronze = (
    raw_data_streaming
    .withColumn("_ingest_timestamp", F.current_timestamp())
    .withColumn("_source", F.lit("delta_landing"))
    .withColumn("_source_file", F.col("_metadata.file_path"))
)

# Process available new commits, then exit so the job can finish.
query = (
    data_bronze.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", data_checkpoint_location)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(bronze_table)
)

query.awaitTermination()

